In [1]:
import sys
import pandas as pd
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    download_etf_prices,
    to_monthly_prices,
    to_monthly_returns,
    download_ff5_monthly,
    merge_etf_ff5,
)



In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /Users/phamthanh/Documents/Project/Financial Market/factorlens_etf_style_drift


### Download ETF adjusted close prices

In [4]:
prices_daily = download_etf_prices(
    start="2015-01-01",
    save_path=RAW_DIR / "etf_prices_daily.csv",
)

print(prices_daily.shape)
prices_daily.tail()

Saved daily prices → /Users/phamthanh/Documents/Project/Financial Market/factorlens_etf_style_drift/data/raw/etf_prices_daily.csv
(2849, 9)


Ticker,IWM,MTUM,QQQ,QUAL,SPY,USMV,VLUE,VTV,VUG
date,,,,,,,,,
2026-04-27,277.140015,280.750000,664.229980,207.190002,715.169983,93.570000,163.529999,203.460007,83.489998
2026-04-28,273.910004,276.209991,657.549988,205.910004,711.690002,93.779999,162.110001,203.500000,82.769997
2026-04-29,272.079987,277.709991,661.570007,205.759995,711.580017,93.879997,164.970001,203.589996,82.680000
2026-04-30,277.970001,283.980011,667.739990,207.250000,718.659973,94.639999,167.070007,206.779999,83.169998
2026-05-01,279.279999,285.089996,674.150024,207.160004,720.650024,94.580002,168.309998,205.949997,83.860001


### Resample to monthly and compute returns

In [5]:
prices_monthly = to_monthly_prices(prices_daily)
prices_monthly.to_csv(RAW_DIR / "etf_prices_monthly.csv")
print("Monthly prices saved:", prices_monthly.shape)

returns_monthly = to_monthly_returns(prices_daily)
returns_monthly.to_csv(PROCESSED_DIR / "etf_returns_monthly.csv")
print("Monthly returns saved:", returns_monthly.shape)
returns_monthly.tail()

Monthly prices saved: (137, 9)
Monthly returns saved: (137, 9)


Ticker,IWM,MTUM,QQQ,QUAL,SPY,USMV,VLUE,VTV,VUG
date,,,,,,,,,
2026-01-31,0.054802,0.022292,0.012306,0.019132,0.014738,0.008815,0.076428,0.045866,-0.012934
2026-02-28,0.006778,-0.011450,-0.023445,0.012054,-0.008642,0.029687,0.024528,0.037597,-0.042945
2026-03-31,-0.049611,-0.050423,-0.048383,-0.061701,-0.049380,-0.047933,-0.052945,-0.048130,-0.051201
2026-04-30,0.120847,0.183299,0.156901,0.080496,0.105053,0.020487,0.174977,0.053925,0.142471
2026-05-31,0.004713,0.003909,0.009600,-0.000434,0.002769,-0.000634,0.007422,-0.004014,0.008296


### Download Fama-French 5-Factor monthly data

In [6]:
ff5 = download_ff5_monthly(
    save_path=RAW_DIR / "ff5_monthly.csv",
)

print(ff5.shape)
ff5.tail()

Saved FF5 monthly factors → /Users/phamthanh/Documents/Project/Financial Market/factorlens_etf_style_drift/data/raw/ff5_monthly.csv
(752, 6)


,Mkt-RF,SMB,HML,RMW,CMA,RF
date,,,,,,
2025-10-31,0.0196,-0.0131,-0.0310,-0.0524,-0.0403,0.0037
2025-11-30,-0.0013,0.0147,0.0376,0.0144,0.0068,0.0030
2025-12-31,-0.0036,-0.0022,0.0242,0.0040,0.0037,0.0034
2026-01-31,0.0103,0.0326,0.0372,0.0182,0.0183,0.0030
2026-02-28,-0.0117,0.0063,0.0283,0.0162,0.0507,0.0028


### Merge ETF returns with FF5 factors

In [7]:
merged = merge_etf_ff5(returns_monthly, ff5)
merged.to_csv(PROCESSED_DIR / "etf_ff5_merged.csv")
print("Merged dataset saved:", merged.shape)
merged.tail()

Merged dataset saved: (1197, 10)


,date,ticker,return,mkt_rf,smb,hml,rmw,cma,rf,excess_return
1192,2025-10-31,VUG,0.040116,0.0196,-0.0131,-0.0310,-0.0524,-0.0403,0.0037,0.036416
1193,2025-11-30,VUG,-0.016057,-0.0013,0.0147,0.0376,0.0144,0.0068,0.0030,-0.019057
1194,2025-12-31,VUG,-0.005055,-0.0036,-0.0022,0.0242,0.0040,0.0037,0.0034,-0.008455
1195,2026-01-31,VUG,-0.012934,0.0103,0.0326,0.0372,0.0182,0.0183,0.0030,-0.015934
1196,2026-02-28,VUG,-0.042945,-0.0117,0.0063,0.0283,0.0162,0.0507,0.0028,-0.045745
